# 流式输出

使用 ChatGPT 或者 Claude 这样的聊天机器人网站，你会发现它们在回答问题的时候，答案是一个字一个字“打”出来的，就仿佛聊天机器人在说话一样，非常生动，交互性很好。

其实，大模型本来就是一个 token 一个 token 地生成回答的：先生成第一个 token，再根据它生成第二个。但是之前我们调用 API 时，服务器要一直等到整个回答都生成完，才一次性发送回来。而**流式输出**（streaming）是让服务器每生成一小段，就马上发一小段过来。

两种方式生成回答花的总时间差不多，但流式输出能让我们**更早看到第一个字**，感觉上快了很多。

In [1]:
import json
import os
import uuid
from abc import ABC, abstractmethod
from enum import Enum
from pathlib import Path
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

## 消息类

### 角色

In [2]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

### 抽象类

In [3]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    @classmethod
    def from_dict(cls, data: dict) -> Message:
        classes = {
            MsgType.SYSTEM.value: SystemMessage,
            MsgType.USER.value: UserMessage,
            MsgType.ASSISTANT.value: AssistantMessage,
        }
        return classes[data["type"]](data["content"])

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

### 系统消息类

In [4]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

### 用户消息类

In [5]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

### 助手消息类

In [6]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

In [7]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

## 记忆类

In [8]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, message: Message):
        self.messages.append(message)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        self.note("summary", self.history())

    def token_count(self):
        return sum(len(msg.summary() or "") for msg in self.messages)

    @staticmethod
    def note(label: str, text: str):
        lines = str(text).splitlines() or [""]
        print(f"  * {label}:")
        for line in lines:
            print(f"  * {line}")

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [Message.from_dict(m) for m in data["messages"]]
        return memory

## 记忆仓库类

In [9]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

## 大模型调用器

### 基础类

`LLMProvider` 多了一个抽象方法 `stream`。它的参数和 `call` 一样，但不是返回一条完整的消息，而是一个生成器，一段一段地给出回答的文字。

In [10]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

    @abstractmethod
    def stream(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        """Yield the reply text chunk by chunk."""
        pass

### OpenAI 调用器

**流式输出**的回答是一段一段到达的，最适合用 Python 的**生成器**（generator）来处理。

普通函数用 `return` 一次性返回一个结果。生成器函数用 `yield`：每 `yield` 一次，就交出一个值，然后暂停，等着被要下一个值时再接着运行。用 `for` 循环遍历生成器，就能一个一个地拿到这些值。

我们就使用生成器模式来实现流失输出：Provider 每收到一小段文字，就 `yield` 出来，`Agent` 用 `for` 循环一段一段地接住，马上打印。

---

流式调用 **OpenAI** 的 API 很简单：在参数里加上 `stream=True`。API 返回的就不再是一个完整的结果，而是一连串的小数据块（chunk）。每个数据块里的 `choices[0].delta.content` 是新生成的那一小段文字。`delta` 的意思是“增量”，也就是“比上一次多出来的部分”。

有些数据块里没有文字（比如最后一个表示结束的数据块），所以要先检查一下，有文字才 `yield`。

---

我们在 OpenAI 调用器里添加流失输出 `stream` 方法：

* 流失调用 OpenAI API；
* 采用生成器方式实时输出结果。

In [11]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    def stream(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages),
            stream=True
        )
        for chunk in self.client.chat.completions.create(**kwargs):
            if chunk.choices and chunk.choices[0].delta.content:
                yield chunk.choices[0].delta.content

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        return AssistantMessage(message.content or "")

### Claude 调用器

流失调用 **Claude** 的 API 稍有不同：用 `client.messages.stream(...)` 打开一个流，它的 `text_stream` 会直接一段一段地给出文字，不用我们自己去数据块里找。

这里用了 `with` 语句：它保证不管中间发生什么，流用完以后都会被正确地关闭，就像用 `with open(...)` 打开文件一样。

两家的写法不同，但对外都是同一个 `stream` 方法。这又一次体现了 Provider 这层外壳的好处。

In [12]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    def stream(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        with self.client.messages.stream(**kwargs) as stream:
            for text in stream.text_stream:
                yield text

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        return AssistantMessage(text)

## 智能体类

智能体类需要有对应的修改：多了一个开关 `stream`，并把“调用大模型并显示回答”这件事放进了新方法 `reply` 里：

- 不用流式输出时，和以前一样，调用 `call`，打印完整的回答。
- 用流式输出时，用 `for` 循环遍历 `stream` 给出的每一小段文字，马上打印出来，同时把它们存进列表 `chunks`。最后把所有片段拼起来，变成一条完整的 `AssistantMessage`，存进记忆。

打印时用了两个参数：

- `end=""`：`print` 默认会在最后换行，这里不换行，让文字接在一起显示。
- `flush=True`：`print` 有时会先把文字攒在缓冲区里，攒够了再显示。`flush=True` 让每一小段文字立刻显示出来。

`chat` 里只改了一行：把原来的 `self.provider.call` 换成了 `self.reply`。

In [13]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage, stream: bool = False):
        self.provider = provider
        self.stream = stream
        self.system = system
        self.store = MemoryStore()
        self.memory = Memory(provider)

    def reply(self, system: SystemMessage, messages: List[Message]) -> AssistantMessage:
        if not self.stream:
            response = self.provider.call(system, messages)
            print(f"AI: {response.summary()}")
            return response

        print("AI: ", end="", flush=True)
        chunks = []
        for text in self.provider.stream(system, messages):
            print(text, end="", flush=True)
            chunks.append(text)
        print()
        return AssistantMessage("".join(chunks))

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None
        self.memory.note("memory", f"#{self.memory.id}")
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                self.memory.note("new memory", f"#{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    self.memory.note("memory", f"{message} not found")
                else:
                    self.memory = memory
                    self.memory.note(f"loaded memory #{memory.id}", memory.history())
                continue

            user_message = UserMessage(message)
            response = self.reply(system or self.system, self.memory.messages + [user_message])
            self.memory.add(user_message)
            self.memory.add(response)
            self.store.save(self.memory)

## 大模型调用

打开流式输出，问两个问题。注意看回答是怎么出现在屏幕上的：

In [14]:
provider = OpenAIProvider()
agent = Agent(provider, system_message, stream=True)
agent.chat(["用三句话介绍一下什么是 Agent。", "再用一句话总结一下。"])

  * memory:
  * #9da0f654

YOU: 用三句话介绍一下什么是 Agent。
AI: Agent（智能体）可以理解为：**能感知环境、做出决策，并采取行动**来完成目标的 AI 系统。  
和只会“回答问题”的大模型不同，Agent 通常还能**调用工具、规划步骤、执行任务并根据结果调整策略**。  
简单说，Agent 就像一个“会自己做事的 AI 助手”，不只是聊天，还能帮你真正完成工作。

YOU: 再用一句话总结一下。
AI: Agent 就是一个**能自主思考、调用工具并执行任务的 AI 助手**。
  * summary:
  * [User]: Historical Summary: 用户先要求用三句话介绍 Agent，助手解释 Agent 是能感知环境、做决策并采取行动完成目标的 AI 系统，还能调用工具、规划和调整策略；随后用户要求再用一句话总结。
  * [Assistant]: Agent 就是一个**能自主思考、调用工具并执行任务的 AI 助手**。

YOU: /quit


## 动手试试

- 把 `stream=True` 改成 `stream=False`，再运行一次，对比一下两种方式的感觉。
- 用 `time.time()` 记录下提问的时间，再分别记下**第一段文字到达**的时间和**全部文字到达**的时间，看看流式输出到底让你早看到了多少。

## 小结

- 大模型本来就是一个 token 一个 token 地生成回答。**流式输出**让服务器边生成边发送，用户能更早看到回答。
- Python 的**生成器**用 `yield` 一个一个地交出值，非常适合处理流式数据。
- OpenAI 用 `stream=True`，从每个数据块的 `delta` 里取出新文字；Claude 用 `messages.stream` 和 `text_stream`。这些差别都藏在 Provider 的 `stream` 方法里。
- `Agent` 用 `print(..., end="", flush=True)` 即时显示每一段文字，最后把所有片段拼成完整的回答存进记忆。

在第二部分里，我们着力于给智能体添加了维持记忆的能力。并在最后一章演示了流失输出的实现方法。